# HippoVoice — Mac Runner (Apple Silicon)

Runs on M1/M2/M3/M4 via MLX. No CUDA required.

## Setup (one-time)
```
pip install mlx-lm chromadb networkx sentence-transformers \
    soundfile jiwer openai-whisper pytest pytest-mock datasets
```
Or just run Step 1 below.

## Expected results
| Section | Expected |
|---------|----------|
| CPU tests | All green |
| Memory walkthrough | Max cancer memories rank above noise |
| LLM smoke test | Response contains "4" |
| Signal/noise benchmark | HippoVoice <10% noise, NaiveRAG >40% |
| LoCoMo benchmark | >65% QA accuracy (Mem0 baseline ~65%) |
| Voice test | Transcription + text response (TTS optional) |

In [ ]:
# ── STEP 1: Install deps (run once, no restart needed on Mac) ─────────────────
import subprocess, sys
def pip(*args): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('mlx-lm')                          # Apple Silicon LLM inference
pip('openai-whisper')                  # STT (runs on MPS)
pip('chromadb', 'networkx')
pip('sentence-transformers>=2.7.0')
pip('soundfile', 'jiwer', 'datasets')
pip('pytest', 'pytest-mock')

import numpy as np
print(f'numpy {np.__version__}  OK')
print('Done — no restart needed.')

In [ ]:
# ── STEP 2: Repo path + auto-reload ──────────────────────────────────────────
# Run from the hippovoice repo root, or set REPO_DIR below.
import os, sys

REPO_DIR = os.path.abspath('.')   # adjust if needed
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

%load_ext autoreload
%autoreload 2

import numpy as np
print(f'numpy {np.__version__}')
print(f'repo: {REPO_DIR}')
print('Ready.')

## 1. CPU Tests

In [ ]:
!pytest tests/test_scorer.py tests/test_extractor.py tests/test_decay.py tests/test_context.py -v --tb=short

In [ ]:
# Salience sanity check
from memory.scorer import compute_salience

checks = [
    ('fear fresh',       compute_salience(1.0, {'label': 'fear',    'intensity': 0.95}, 0,  0), '> 3.0'),
    ('neutral 45 turns', compute_salience(1.0, {'label': 'neutral', 'intensity': 0.01}, 0, 45), '< 0.25'),
    ('fear 45 turns',    compute_salience(1.0, {'label': 'fear',    'intensity': 0.90}, 0, 45), '> 0.25'),
]
for label, val, expectation in checks:
    print(f'{label:20s}: {val:.4f}  ({expectation})')

## 2. Memory Core + Store Tests

In [ ]:
!pytest tests/test_store.py tests/test_retriever.py -v --tb=short

In [ ]:
# Manual memory walkthrough
from memory.store import HippoMemory
from memory.retriever import hippo_retrieve

mem = HippoMemory()
entries = [
    ('My dog Max got diagnosed with cancer. I am devastated.',       'sadness', 0.9),
    ('The weather was cloudy today.',                                'neutral', 0.05),
    ('Max had his first chemo session. I held him the whole time.',  'sadness', 0.85),
    ('I had cereal for breakfast.',                                  'neutral', 0.05),
]
for i, (content, label, intensity) in enumerate(entries):
    mem.add({'content': content, 'emotion': {'label': label, 'intensity': intensity},
             'base_weight': 1.0, 'recall_count': 0, 'turn_created': i})

print(f'Stored {mem.count()} memories\n')
results = hippo_retrieve('tell me about Max', mem, mem.graph, current_turn=4, top_k=4)
for r in results:
    print(f'  [{r["current_salience"]:.3f}] {r["content"]}')

## 3. Load LLM (Qwen3-4B via MLX, 4-bit — ~2.5GB RAM)

In [ ]:
from llm.client import LLMClient

# mlx-community/Qwen3-4B-4bit is a pre-quantized MLX model (~2.5GB download)
llm = LLMClient(model_name='mlx-community/Qwen3-4B-4bit')
print(f'Backend: {llm._backend}  device: {llm.device}')

In [ ]:
# Smoke test
resp = llm.generate(
    system='Answer in one sentence.',
    messages=[{'role': 'user', 'content': 'What is 2 + 2?'}],
    max_tokens=30
)
print('Response:', resp)
assert '4' in resp
print('LLM working')

## 4. Signal/Noise Benchmark (Core Research Claim)

45-turn synthetic conversation: 22 high-salience signal turns (fear/sadness 0.7–0.95)
interleaved with 22 low-salience noise turns (neutral 0.01).

**Pass criteria:**
- HippoVoice noise rate < 10% — salience decay filters noise out
- NaiveRAG noise rate > 40% — flat cosine retrieval can't distinguish

In [ ]:
from pipeline import HippoVoicePipeline
from baselines.naive_rag import NaiveRAG
from benchmarks.signal_noise.run import run_signal_noise_benchmark

print('Running signal/noise benchmark (~2 min on M4)...\n')

hippo = HippoVoicePipeline(llm_client=llm, text_only=True)
naive = NaiveRAG()

hippo_result = run_signal_noise_benchmark(hippo, 'HippoVoice')
naive_result  = run_signal_noise_benchmark(naive,  'NaiveRAG')

print('=' * 50)
print(f'HippoVoice  noise rate: {hippo_result["noise_rate"]:.1%}  (signal={hippo_result["signal_count"]}, noise={hippo_result["noise_count"]})')
print(f'Naive RAG   noise rate: {naive_result["noise_rate"]:.1%}  (signal={naive_result["signal_count"]}, noise={naive_result["noise_count"]})')
print('=' * 50)
print(f'HippoVoice < 10% noise: {"PASS" if hippo_result["noise_rate"] < 0.10 else "FAIL"}')
print(f'Naive RAG  > 40% noise: {"PASS" if naive_result["noise_rate"]  > 0.40 else "FAIL"}')

## 5. LoCoMo Benchmark

30 long-term conversations with QA ground truth. Mem0 baseline ~65%.

In [ ]:
from benchmarks.locomo.evaluate import run_locomo
from pipeline import HippoVoicePipeline

pipe = HippoVoicePipeline(llm_client=llm, text_only=True)
result = run_locomo(pipeline=pipe, num_conversations=30)

print(f'LoCoMo accuracy: {result["accuracy"]:.1%}  ({result["correct"]}/{result["total"]})')
print(f'Mem0 baseline:   ~65%')
print(f'Above baseline:  {"YES" if result["accuracy"] > 0.65 else "NO"}')

## 6. Voice Test (Whisper STT + text response)

Records via mic → transcribes with Whisper → extracts memories → LLM response.
TTS not included here (Fish Speech needs CUDA); response is printed as text.

In [ ]:
# Record audio via sounddevice (install if missing: pip install sounddevice)
import sounddevice as sd
import soundfile as sf
import numpy as np

AUDIO_PATH = '/tmp/hippo_input.wav'
SAMPLE_RATE = 16000
DURATION = 5

print(f'Recording {DURATION}s — speak now...')
audio = sd.rec(int(DURATION * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
sd.wait()
sf.write(AUDIO_PATH, audio, SAMPLE_RATE)
print(f'Saved to {AUDIO_PATH}')

In [ ]:
import whisper
import numpy as np

# Load Whisper (base model ~150MB, runs on MPS)
stt = whisper.load_model('base')
result_stt = stt.transcribe(AUDIO_PATH)
transcript = result_stt['text'].strip()
print(f'Transcript: "{transcript}"')

# Dummy audio embedding (prosody fusion — full Canary embeddings not needed for text benchmarks)
audio_emb = np.zeros(512)

from memory.extractor import extract_turn
from memory.store import HippoMemory
from memory.retriever import hippo_retrieve
from llm.context import build_system_prompt, BASE_COMPANION_PROMPT

mem = HippoMemory()
memories = extract_turn(transcript, audio_emb, llm)
print(f'\nExtracted {len(memories)} memories:')
for m in memories:
    print(f'  [{m["emotion"]["label"]} {m["emotion"]["intensity"]:.2f}] {m["content"]}')

system = build_system_prompt([], BASE_COMPANION_PROMPT)
response_text = llm.generate(
    system=system,
    messages=[{'role': 'user', 'content': transcript}],
    max_tokens=100
)
print(f'\nResponse: "{response_text}"')

## 7. Save Results

In [ ]:
import json, datetime

out = {
    'timestamp': datetime.datetime.now().isoformat(),
    'platform': 'mac-apple-silicon',
    'llm': 'Qwen3-4B-MLX-4bit',
    'signal_noise': {k: v for k, v in hippo_result.items() if k != 'retrieved'} if 'hippo_result' in dir() else 'not run',
    'naive_rag':    {k: v for k, v in naive_result.items()  if k != 'retrieved'} if 'naive_result'  in dir() else 'not run',
    'locomo':       {'accuracy': result['accuracy']} if 'result' in dir() else 'not run',
}

out_path = 'hippovoice_results_mac.json'
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f'Saved to {out_path}')
print(json.dumps(out, indent=2))